In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# Cell 1: Install dependencies
!pip install -q tiktoken datasets tqdm

In [3]:
# Cell 2: Data Preparation
import os
import numpy as np
import tiktoken
from datasets import load_dataset
from tqdm import tqdm

# Load a small subset of TinyStories
dataset = load_dataset("roneneldan/TinyStories", split='train[:5000]') # 5000 stories for speed
split_dataset = dataset.train_test_split(test_size=0.1)

enc = tiktoken.get_encoding("gpt2")

def process(example):
    ids = enc.encode_ordinary(example['text']) 
    ids.append(enc.eot_token) # append End of Text token
    return {'ids': ids, 'len': len(ids)}

# Tokenize
tokenized = split_dataset.map(process, remove_columns=['text'], num_proc=4)

# Save to binary files
for split, dset in tokenized.items():
    arr_len = np.sum(dset['len'])
    filename = f'{split}.bin'
    arr = np.memmap(filename, dtype=np.uint16, mode='w+', shape=(arr_len,))
    
    idx = 0
    for example in tqdm(dset['ids'], desc=f'writing {filename}'):
        arr[idx : idx + len(example)] = example
        idx += len(example)
    arr.flush()

print("Data preparation complete! train.bin and test.bin created.")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/4500 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/500 [00:00<?, ? examples/s]

writing test.bin: 100%|██████████| 500/500 [00:00<00:00, 8968.97it/s]

Data preparation complete! train.bin and test.bin created.


In [4]:
# Cell 3: The Model
import torch
import torch.nn as nn
from torch.nn import functional as F

# Hyperparameters
batch_size = 32 
block_size = 256 # context length
max_iters = 2000
eval_interval = 200
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embd = 128
n_head = 4
n_layer = 4

class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   
        q = self.query(x) 
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) 
        wei = F.softmax(wei, dim=-1) 
        # perform the weighted aggregation of the values
        v = self.value(x) 
        out = wei @ v 
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class NanoGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(50257, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, 50257)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) 
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) 
        x = tok_emb + pos_emb 
        x = self.blocks(x) 
        x = self.ln_f(x) 
        logits = self.lm_head(x) 

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

In [8]:
# Cell 4: Multi-GPU Training Engine
import numpy as np

# 1. Load data
train_data = np.memmap('train.bin', dtype=np.uint16, mode='r')
val_data = np.memmap('test.bin', dtype=np.uint16, mode='r')

# 2. Double batch size for 2 GPUs
batch_size = 64 

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+block_size+1]).astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

# 3. Initialize and Wrap Model
model = NanoGPT()

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    # This splits the 64 batch into 32 per GPU
    model = nn.DataParallel(model)

model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# 4. Training Loop
for iter in range(max_iters):
    if iter % eval_interval == 0:
        print(f"step {iter}: computing loss...")
        
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    loss = loss.mean()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % 500 == 0:
        # Note: we use .mean() in case DataParallel returns a vector of losses
        print(f"Iter {iter}, Loss: {loss.mean().item():.4f}")

# 5. Save correctly (handling the DataParallel wrapper)
if isinstance(model, nn.DataParallel):
    torch.save(model.module.state_dict(), 'nanogpt.pt')
else:
    torch.save(model.state_dict(), 'nanogpt.pt')

print("Training finished and model saved!")

Using 2 GPUs!
step 0: computing loss...
Iter 0, Loss: 11.0257
step 200: computing loss...
step 400: computing loss...
Iter 500, Loss: 4.2520
step 600: computing loss...
step 800: computing loss...
step 1000: computing loss...
Iter 1000, Loss: 3.7747
step 1200: computing loss...
step 1400: computing loss...
Iter 1500, Loss: 3.4807
step 1600: computing loss...
step 1800: computing loss...
Training finished and model saved!
